# AutoML Framework API Demo (Final Fixed Version)

This notebook demonstrates how to:
1. Login as admin to the AutoML Framework API
2. Create a time series dataset
3. Upload the dataset
4. Create a time series forecasting project/experiment
5. Monitor the experiment progress
6. Retrieve results

## Prerequisites
- AutoML Framework API running on http://localhost:8000
- Admin credentials: username='admin', password='secret'

In [ ]:
# Import required libraries
import requests
import pandas as pd
import numpy as np
import json
import time
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

# Set up plotting
try:
    plt.style.use('seaborn-v0_8')
except:
    plt.style.use('seaborn')
sns.set_palette("husl")

print("Libraries imported successfully!")

## 1. API Configuration and Authentication

In [6]:
#!/usr/bin/env python3
"""
AutoML Framework API Demo - Fixed Version
This script demonstrates the complete workflow with proper dataset path handling.
"""

import requests
import pandas as pd
import numpy as np
import json
import time
from datetime import datetime, timedelta

# Configuration
API_BASE_URL = "http://localhost:8000"
ADMIN_USERNAME = "admin"
ADMIN_PASSWORD = "secret"


# Global session for API calls
session = requests.Session()

def login(username: str, password: str) -> str:
    """Login to the API and return access token."""
    login_url = f"{API_BASE_URL}/api/v1/auth/login"
    
    login_data = {
        "username": username,
        "password": password
    }
    
    headers = {
        "Content-Type": "application/x-www-form-urlencoded"
    }
    
    response = requests.post(login_url, data=login_data, headers=headers)
    
    if response.status_code == 200:
        token_data = response.json()
        access_token = token_data["access_token"]
        print(f"✅ Successfully logged in as {username}")
        return access_token
    else:
        print(f"❌ Login failed: {response.status_code}")
        print(f"Error: {response.text}")
        return None

def generate_time_series_data(n_points=800, start_date='2020-01-01'):
    """Generate synthetic time series data for demonstration."""
    
    # Create date range
    dates = pd.date_range(start=start_date, periods=n_points, freq='D')
    
    # Generate base trend
    trend = np.linspace(100, 200, n_points)
    
    # Add seasonal component (yearly cycle)
    seasonal = 20 * np.sin(2 * np.pi * np.arange(n_points) / 365.25)
    
    # Add weekly pattern
    weekly = 10 * np.sin(2 * np.pi * np.arange(n_points) / 7)
    
    # Add noise
    noise = np.random.normal(0, 5, n_points)
    
    # Combine components
    values = trend + seasonal + weekly + noise
    
    # Add some external features
    temperature = 20 + 15 * np.sin(2 * np.pi * np.arange(n_points) / 365.25) + np.random.normal(0, 2, n_points)
    day_of_week = np.arange(n_points) % 7
    month = dates.month
    
    # Create DataFrame
    df = pd.DataFrame({
        'date': dates,
        'value': values,
        'temperature': temperature,
        'day_of_week': day_of_week,
        'month': month,
        'is_weekend': (day_of_week >= 5).astype(int)
    })
    
    return df

def upload_dataset(file_path: str, name: str, description: str, token: str):
    """Upload dataset to the AutoML Framework."""
    upload_url = f"{API_BASE_URL}/api/v1/datasets/upload"
    
    # Prepare files and data for upload
    with open(file_path, 'rb') as f:
        files = {'file': (file_path, f, 'text/csv')}
        data = {
            'name': name,
            'description': description
        }
        
        headers = {'Authorization': f'Bearer {token}'}
        
        response = requests.post(upload_url, files=files, data=data, headers=headers)
    
    return response

def create_experiment(name: str, dataset_file_path: str, task_type: str, data_type: str, 
                     target_column: str, config: dict, token: str):
    """Create a new experiment."""
    create_url = f"{API_BASE_URL}/api/v1/experiments"
    
    experiment_data = {
        "name": name,
        "dataset_path": dataset_file_path,  # Use the actual file path from upload response
        "task_type": task_type,
        "data_type": data_type,
        "target_column": target_column,
        "config": config
    }
    
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }
    
    response = requests.post(create_url, json=experiment_data, headers=headers)
    return response


print("🚀 AutoML Framework API Demo - Fixed Version")
print("=" * 60)

# Step 1: Login
print("\n1. 🔐 Authenticating...")
access_token = login(ADMIN_USERNAME, ADMIN_PASSWORD)
if not access_token:
    print("❌ Authentication failed. Exiting.")
    

# Step 2: Generate data
print("\n2. 📊 Generating synthetic time series data...")
ts_data = generate_time_series_data(n_points=800)
print(f"✅ Generated dataset with {len(ts_data)} data points")
print(f"Date range: {ts_data['date'].min()} to {ts_data['date'].max()}")

# Step 3: Save and upload dataset
print("\n3. 📤 Uploading dataset...")
csv_filename = "time_series_demo_data.csv"
ts_data.to_csv(csv_filename, index=False)
print(f"💾 Saved dataset to {csv_filename}")

upload_response = upload_dataset(
    file_path=csv_filename,
    name="Time Series Demo Dataset",
    description="Synthetic time series data with trend, seasonality, and external features for forecasting demo",
    token=access_token
)

print(f"Upload response status: {upload_response.status_code}")

if upload_response.status_code == 200:
    upload_data = upload_response.json()
    dataset_id = upload_data['dataset_id']
    dataset_file_path = upload_data['file_path']  # THIS IS THE KEY FIX!
    print(f"✅ Dataset uploaded successfully!")
    print(f"Dataset ID: {dataset_id}")
    print(f"File Path: {dataset_file_path}")  # This is what we need for the experiment
    print(f"Filename: {upload_data['filename']}")
    print(f"Size: {upload_data['size_bytes']} bytes")
else:
    print(f"❌ Dataset upload failed: {upload_response.status_code}")
    print(f"Error: {upload_response.text}")
    

# Step 4: Create experiment
print("\n4. 🧪 Creating time series forecasting experiment...")
experiment_config = {
    "forecast_horizon": 30,
    "lookback_window": 60,
    "validation_split": 0.2,
    "max_trials": 3,  # Small number for demo
    "max_epochs": 10,  # Small number for demo
    "features": {
        "date_column": "date",
        "value_column": "value",
        "external_features": ["temperature", "day_of_week", "month", "is_weekend"]
    },
    "model_types": ["lstm"],
    "optimization_metric": "mae",
    "early_stopping": {
        "patience": 5,
        "min_delta": 0.001
    }
}

experiment_response = create_experiment(
    name="Time Series Forecasting Demo",
    dataset_file_path=dataset_file_path,  # Use the actual file path from upload
    task_type="time_series_forecasting",
    data_type="time_series",
    target_column="value",
    config=experiment_config,
    token=access_token
)

print(f"Experiment response status: {experiment_response.status_code}")

if experiment_response.status_code == 200:
    experiment_data = experiment_response.json()
    experiment_id = experiment_data['id']
    print(f"✅ Experiment created successfully!")
    print(f"Experiment ID: {experiment_id}")
    print(f"Name: {experiment_data['name']}")
    print(f"Status: {experiment_data['status']}")
    print(f"Created at: {experiment_data['created_at']}")
else:
    print(f"❌ Experiment creation failed: {experiment_response.status_code}")
    print(f"Error: {experiment_response.text}")
    

# Step 5: List experiments
print("\n5. 📋 Listing all experiments...")
list_url = f"{API_BASE_URL}/api/v1/experiments"
headers = {"Authorization": f"Bearer {access_token}"}
list_response = requests.get(list_url, headers=headers)

if list_response.status_code == 200:
    experiments_data = list_response.json()
    experiments = experiments_data.get('experiments', [])
    total = experiments_data.get('total', 0)
    
    print(f"✅ Found {total} experiment(s)")
    
    if experiments:
        print(f"\n📊 Experiments Summary:")
        print("-" * 80)
        print(f"{'ID':<20} {'Name':<30} {'Status':<12} {'Created':<20}")
        print("-" * 80)
        
        for exp in experiments:
            exp_id = exp['id'][:18] + '...' if len(exp['id']) > 20 else exp['id']
            exp_name = exp['name'][:28] + '...' if len(exp['name']) > 30 else exp['name']
            exp_status = exp['status']
            exp_created = exp['created_at'][:19] if exp['created_at'] else 'N/A'
            
            print(f"{exp_id:<20} {exp_name:<30} {exp_status:<12} {exp_created:<20}")
        
        print("-" * 80)
else:
    print(f"❌ Failed to list experiments: {list_response.status_code}")
    print(f"Error: {list_response.text}")

print("\n🎉 Demo completed successfully!")
print("\n" + "=" * 60)
print("KEY FIX APPLIED:")
print("=" * 60)
print("✅ Used actual file_path from upload response instead of constructing path")
print(f"✅ Correct path format: {dataset_file_path}")
print("✅ This ensures the experiment can find the uploaded dataset file")
print("=" * 60)



🚀 AutoML Framework API Demo - Fixed Version

1. 🔐 Authenticating...
✅ Successfully logged in as admin

2. 📊 Generating synthetic time series data...
✅ Generated dataset with 800 data points
Date range: 2020-01-01 00:00:00 to 2022-03-10 00:00:00

3. 📤 Uploading dataset...
💾 Saved dataset to time_series_demo_data.csv
Upload response status: 200
✅ Dataset uploaded successfully!
Dataset ID: 31091557-a7de-4465-8426-8aa6483a6713
File Path: data/uploads/31091557-a7de-4465-8426-8aa6483a6713.csv
Filename: time_series_demo_data.csv
Size: 43528 bytes

4. 🧪 Creating time series forecasting experiment...
Experiment response status: 200
✅ Experiment created successfully!
Experiment ID: dfd700cf-bc9e-4a4b-bd82-f2d99dad9e31
Name: Time Series Forecasting Demo
Status: created
Created at: 2025-08-17T18:59:32.872247

5. 📋 Listing all experiments...
✅ Found 3 experiment(s)

📊 Experiments Summary:
--------------------------------------------------------------------------------
ID                   Name     

In [7]:
def get_experiment_status(experiment_id: str):
    """Get current experiment status."""
    status_url = f"{API_BASE_URL}/api/v1/experiments/{experiment_id}"
    response = session.get(status_url)
    return response

def monitor_experiment(experiment_id: str, max_wait_time: int = 300, check_interval: int = 10):
    """Monitor experiment progress until completion or timeout."""
    start_time = time.time()
    
    print(f"🔍 Monitoring experiment {experiment_id}...")
    print(f"Max wait time: {max_wait_time} seconds")
    print(f"Check interval: {check_interval} seconds")
    print("\n" + "="*60)
    
    while time.time() - start_time < max_wait_time:
        response = get_experiment_status(experiment_id)
        
        if response.status_code == 200:
            data = response.json()
            status = data['status']
            progress = data.get('progress', {})
            
            elapsed_time = int(time.time() - start_time)
            timestamp = datetime.now().strftime("%H:%M:%S")
            
            print(f"[{timestamp}] Status: {status.upper()} | Elapsed: {elapsed_time}s")
            
            if progress:
                for key, value in progress.items():
                    if isinstance(value, float):
                        print(f"  {key}: {value:.2%}")
                    else:
                        print(f"  {key}: {value}")
            
            if status in ['completed', 'failed', 'cancelled']:
                print(f"\n🏁 Experiment {status.upper()}!")
                return data
            
            print("-" * 40)
            time.sleep(check_interval)
        else:
            print(f"❌ Failed to get experiment status: {response.status_code}")
            break
    
    print(f"\n⏰ Monitoring timeout reached ({max_wait_time}s)")
    return None

if experiment_id:
    # Monitor the experiment (with shorter timeout for demo)
    final_status = monitor_experiment(experiment_id, max_wait_time=180, check_interval=15)
    
    if final_status:
        print(f"\n📊 Final Experiment Status:")
        print(f"ID: {final_status['id']}")
        print(f"Name: {final_status['name']}")
        print(f"Status: {final_status['status']}")
        print(f"Created: {final_status['created_at']}")
        if final_status.get('completed_at'):
            print(f"Completed: {final_status['completed_at']}")
        if final_status.get('results'):
            print(f"\n🎯 Results Available: Yes")
        else:
            print(f"\n🎯 Results Available: No (experiment may still be running)")
else:
    print("⚠️ Skipping experiment monitoring due to creation failure")

🔍 Monitoring experiment dfd700cf-bc9e-4a4b-bd82-f2d99dad9e31...
Max wait time: 180 seconds
Check interval: 15 seconds



NameError: name 'session' is not defined

In [8]:
def get_experiment_results(experiment_id: str):
    """Get detailed experiment results."""
    results_url = f"{API_BASE_URL}/api/v1/experiments/{experiment_id}/results"
    response = session.get(results_url)
    return response

if experiment_id:
    print(f"📈 Retrieving experiment results...")
    results_response = get_experiment_results(experiment_id)
    
    if results_response.status_code == 200:
        results_data = results_response.json()
        print(f"✅ Results retrieved successfully!")
        
        # Display results summary
        if 'best_model' in results_data:
            best_model = results_data['best_model']
            print(f"\n🏆 Best Model:")
            print(f"  Architecture: {best_model.get('architecture', 'N/A')}")
            print(f"  Score: {best_model.get('score', 'N/A')}")
            print(f"  Metric: {best_model.get('metric', 'N/A')}")
        
        if 'metrics' in results_data:
            metrics = results_data['metrics']
            print(f"\n📊 Performance Metrics:")
            for metric_name, metric_value in metrics.items():
                if isinstance(metric_value, float):
                    print(f"  {metric_name}: {metric_value:.4f}")
                else:
                    print(f"  {metric_name}: {metric_value}")
        
        if 'predictions' in results_data:
            predictions = results_data['predictions']
            print(f"\n🔮 Predictions: {len(predictions)} data points")
            
            # Convert predictions to DataFrame for visualization
            pred_df = pd.DataFrame(predictions)
            
            # Plot predictions vs actual values
            if 'actual' in pred_df.columns and 'predicted' in pred_df.columns:
                plt.figure(figsize=(12, 6))
                
                plt.subplot(1, 2, 1)
                plt.plot(pred_df['actual'], label='Actual', alpha=0.8)
                plt.plot(pred_df['predicted'], label='Predicted', alpha=0.8)
                plt.title('Time Series Predictions vs Actual')
                plt.xlabel('Time Step')
                plt.ylabel('Value')
                plt.legend()
                plt.grid(True, alpha=0.3)
                
                plt.subplot(1, 2, 2)
                plt.scatter(pred_df['actual'], pred_df['predicted'], alpha=0.6)
                plt.plot([pred_df['actual'].min(), pred_df['actual'].max()], 
                        [pred_df['actual'].min(), pred_df['actual'].max()], 
                        'r--', alpha=0.8)
                plt.title('Predicted vs Actual Values')
                plt.xlabel('Actual Values')
                plt.ylabel('Predicted Values')
                plt.grid(True, alpha=0.3)
                
                plt.tight_layout()
                plt.show()
        
        # Display full results (truncated for readability)
        print(f"\n📋 Full Results Summary:")
        print(json.dumps(results_data, indent=2, default=str)[:2000] + "..." if len(str(results_data)) > 2000 else json.dumps(results_data, indent=2, default=str))
        
    elif results_response.status_code == 404:
        print(f"⚠️ Results not yet available. Experiment may still be running.")
    else:
        print(f"❌ Failed to retrieve results: {results_response.status_code}")
        print(f"Error: {results_response.text}")
else:
    print("⚠️ Skipping results retrieval due to experiment creation failure")

📈 Retrieving experiment results...


NameError: name 'session' is not defined